# Synthetic eval-data generation

Hand-curating golden Q&A doesn't scale. A 1000-question set is a month of work; an LLM can produce one in minutes. **The catch**: synthetic Q&A is biased toward what the *generator* finds important, which is usually whatever's in the first sentence. Filter aggressively and *calibrate* against a small golden set.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
from eval import generate, filter_empty, filter_answer_in_source, filter_question_length
from shared.loaders import load_corpus, load_golden_qa
DOCS = load_corpus()
print(f'{len(DOCS)} docs in the corpus')

## Step 1 — generate one Q&A per doc

In [ ]:
generated = [generate(d.arxiv_id, d.abstract) for d in DOCS]
for g in generated[:3]:
    print(f"[{g['source_id']}] Q: {g['question'][:80]}...")
    print(f"      A: {g['answer'][:80]}...")
    print()

## Step 2 — filter

Three filters: empty, length-sane, answer-grounded-in-source.

In [ ]:
drops = {'empty': 0, 'length': 0, 'unsupported': 0}
kept = []
for q, d in zip(generated, DOCS):
    if not filter_empty(q): drops['empty'] += 1; continue
    if not filter_question_length(q): drops['length'] += 1; continue
    if not filter_answer_in_source(q, d.abstract): drops['unsupported'] += 1; continue
    kept.append(q)
print(f'generated={len(generated)}  kept={len(kept)}  drops={drops}')

## Step 3 — calibrate against the golden set

Compute retrieval recall@3 on the synthetic set and on the golden set with the *same* retriever. If the synthetic recall is much higher, the synthetic questions are too close to the source text — the eval isn't measuring retrieval, it's measuring template adherence.

In [ ]:
import numpy as np
from shared.embedders import cosine_topk, hash_embed
DIMS = 256
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=DIMS, seed=0)

def recall(qs):
    hits = 0
    for q in qs:
        qv = hash_embed([q['question']], dims=DIMS, seed=0)[0]
        idx, _ = cosine_topk(qv, doc_vecs, k=3)
        if q['source_id'] in {doc_ids[i] for i in idx}: hits += 1
    return round(hits / len(qs), 4) if qs else 0.0

golden = [{'question': q.question, 'source_id': q.source_ids[0]} for q in load_golden_qa() if q.source_ids]
print(f'synthetic recall@3 : {recall(kept)}')
print(f'golden    recall@3 : {recall(golden)}')
print(f'calibration gap    : {recall(kept) - recall(golden):+.3f}')

## Takeaways

* Synthetic Q&A is **free** but **biased** — the gap to a small hand-curated golden set is the bias.
* If the gap is < 0.1, the synthetic set is calibrated; use it for regression testing.
* If the gap is > 0.2, refresh the generator prompt — ask for harder questions (multi-hop, paraphrased, requiring inference).
* Always commit the synthetic set (`generated-qa.jsonl`) so it can be diffed in PRs alongside the eval-snapshot.